# Step 2: Java Source Code Translation (Layer 2 & 3 Consolidated)

This notebook simulates and prototypes the code refactoring process on the `working/sonar-stash` project codebase.

This step focuses on:
- Detecting JDK 17 deprecated/removed APIs (Layer 2) using `jdeprscan`.
- Collecting compilation errors under JDK 17 (Layer 3) using `mvn compile`.
- Bundling findings by file and querying local Ollama LLM to refactor using a list of non-overlapping block edits applied from bottom to top.

In [11]:
import sys
import os
import json
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root added to path: {PROJECT_ROOT}")

# Check availability of key imports
from src.tools.maven import MavenProject, MavenPomEditor, MavenRunner
from src.core.tree_sitter_engine import UniversalASTReader
from src.tools.codebase_search_tools import find_code_usages, search_codebase

print("Successfully imported models and tools!")

# Setup paths globally (Adjust PROJECT_PATH to point to your target Java project)
PROJECT_PATH = Path("working/sonar-stash").resolve()
pom_path = PROJECT_PATH / "pom.xml"
ARTIFACTS_DIR = PROJECT_PATH / "artifacts"

review_file = ARTIFACTS_DIR / "reader_review.json"
if not review_file.exists():
    review_file = Path("dependency_focus_scopes.json").resolve()

print(f"Working Project path: {PROJECT_PATH} (exists: {PROJECT_PATH.exists()})")
print(f"Artifacts path: {ARTIFACTS_DIR}")
print(f"Review report path: {review_file} (exists: {review_file.exists()})")

Project root added to path: D:\capstone_project\MYGRATE---Capstone-Project
Successfully imported models and tools!
Working Project path: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash (exists: True)
Artifacts path: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash\artifacts
Review report path: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash\artifacts\reader_review.json (exists: True)


In [12]:
# Load POM upgrade results from POM migration phase
review_data = {}
selected_sol = {}

if review_file.exists():
    with open(review_file, "r", encoding="utf-8") as f:
        review_data = json.load(f)
    review_body = review_data.get("review_upgrade_candidates", {})
    selected_sol = review_body.get("selected_solution", {})
    print("Upgraded Dependency Mappings (from reader_review.json):")
    print(json.dumps(selected_sol, indent=2))
else:
    print("reader_review.json not found!")

Upgraded Dependency Mappings (from reader_review.json):
{
  "org.sonarsource.sonarqube:sonar-plugin-api": "9.3.0.51899",
  "com.github.cliftonlabs:json-simple": "4.0.1",
  "org.asynchttpclient:async-http-client": "3.0.2",
  "com.google.guava:guava": "20.0",
  "org.assertj:assertj-core": "3.26.3",
  "org.mockito:mockito-core": "5.15.2",
  "org.mockito:mockito-junit-jupiter": "5.15.2",
  "org.awaitility:awaitility": "4.1.1",
  "com.github.tomakehurst:wiremock": "2.27.2",
  "org.picocontainer:picocontainer": "2.14.2",
  "com.google.code.gson:gson": "2.12.0"
}


## 1. Consolidated Compilation & Deprecated API Scan

We run JDK 17 compilation to get modern compiler errors and execute `jdeprscan` to capture deprecated/removed API usages. These are aggregated by Java source file.

In [13]:
from src.tools.maven import MavenRunner, MavenProject
from src.tools.jdeprscan_pipeline import run_jdeprscan_pipeline
import re

# 1. Compile project under JDK 17 once to extract initial compile errors
print("Compiling project under JDK 17...")
runner = MavenRunner(target_java_version="17")
compile_res = runner.compile(PROJECT_PATH, clean=True)
print(f"Initial compile status: {compile_res.status}")

# 2. Run jdeprscan pipeline under JDK 17 to detect deprecated/removed APIs (Level 2)
print("Running jdeprscan pipeline on the working codebase...")
updated_jdeprscan_result = run_jdeprscan_pipeline(
    project_path=str(PROJECT_PATH),
    target_release="17",
    logger=lambda msg: print(f"  {msg}")
)

file_findings = {}

# Helper function to extract compile errors from maven stdout/stderr output without using fragile regex
def extract_file_errors(compile_output, project_path):
    findings = []
    for line in compile_output.splitlines():
        if "[ERROR]" in line and ".java" in line:
            parts = line.split(".java")
            if len(parts) >= 2:
                path_part = parts[0].strip()
                if path_part.startswith("[ERROR]"):
                    path_part = path_part[len("[ERROR]"):].strip()
                if path_part.startswith("/") and len(path_part) > 2 and path_part[2] == ":":
                    path_part = path_part[1:]
                
                abs_path = Path(path_part + ".java")
                try:
                    rel_path = abs_path.relative_to(project_path)
                except ValueError:
                    if "src/" in path_part:
                        rel_path = Path("src/" + path_part.split("src/", 1)[1] + ".java")
                    elif "src\\" in path_part:
                        rel_path = Path("src/" + path_part.split("src\\", 1)[1].replace("\\", "/") + ".java")
                    else:
                        rel_path = abs_path
                
                standard_path = str(rel_path).replace("\\", "/")
                details = parts[1].strip()
                
                line_match = re.search(r'\d+', details)
                line_no = line_match.group(0) if line_match else "unknown"
                
                msg = details
                if msg.startswith(":"):
                    msg = msg[1:].strip()
                if msg.startswith("[") and "]" in msg:
                    msg = msg.split("]", 1)[1].strip()
                if line_no != "unknown" and msg.startswith(line_no):
                    msg = msg[len(line_no):].strip()
                if msg.startswith(":"):
                    msg = msg[1:].strip()
                if msg.lower().startswith("error:"):
                    msg = msg[6:].strip()
                if msg.startswith(":"):
                    msg = msg[1:].strip()
                
                findings.append((standard_path, line_no, msg))
    return findings

# Parse errors from JDK 17 compilation
compile_errors_matched = extract_file_errors(compile_res.stdout + "\n" + compile_res.stderr, PROJECT_PATH)
print(f"\nExtracted {len(compile_errors_matched)} compile error(s) under JDK 17:")
for standard_path, line_no, err_msg in compile_errors_matched:
    file_findings.setdefault(standard_path, {"findings": [], "matching_rules": []})
    file_findings[standard_path]["findings"].append(f"Compile Error (Line {line_no}): {err_msg}")
    print(f"  * {standard_path}:{line_no} -> {err_msg}")

# Parse compile errors from jdeprscan b1 step (runs JDK 8 compile as part of scan pipeline)
b1_step = updated_jdeprscan_result.get("steps", {}).get("b1_compile", {})
if b1_step.get("status") == "FAIL":
    b1_error = b1_step.get("error", "")
    b1_matches = extract_file_errors(b1_error, PROJECT_PATH)
    for standard_path, line_no, err_msg in b1_matches:
        file_findings.setdefault(standard_path, {"findings": [], "matching_rules": []})
        file_findings[standard_path]["findings"].append(f"Compile Error (Line {line_no}): {err_msg}")

# Parse jdeprscan B2 project scan warnings/removal items
b2_step = updated_jdeprscan_result.get("steps", {}).get("b2_project_scan", {})
for line in b2_step.get("for_removal", []) + b2_step.get("deprecated", []):
    m = re.match(r"class\s+([^\s]+)\s+uses", line)
    if m:
        class_path = m.group(1)
        rel_path = f"src/main/java/{class_path}.java"
        file_findings.setdefault(rel_path, {"findings": [], "matching_rules": []})
        file_findings[rel_path]["findings"].append(f"Jdeprscan warning: {line}")

# Parse reader review flagged risks
if review_file.exists():
    review_body = review_data.get("review_upgrade_candidates", {})
    risks_text = review_body.get("risks", "")
    candidates = re.findall(r"\b(?:javax\.\w+|com\.sun\.\w+)\b", risks_text)
    for cand in candidates:
        matches = search_codebase(
            project_path=str(PROJECT_PATH),
            regex_pattern=r"import\s+" + cand.replace(".", r"\."),
            file_extensions=["java"]
        )
        for m in matches:
            rel_path = m["file_path"].replace("\\", "/")
            file_findings.setdefault(rel_path, {"findings": [], "matching_rules": []})
            file_findings[rel_path]["findings"].append(f"Reader review flagged risk: usage of {cand}")

# Deduplicate findings per file
for path, info in file_findings.items():
    info["findings"] = list(set(info["findings"]))

# Load rules and match them against findings and codebase contents
rules_file = Path("migration_rules.json")
rules = []
if rules_file.exists():
    with open(rules_file, "r", encoding="utf-8") as f:
        rules = json.load(f).get("rules", [])

for rel_path, info in file_findings.items():
    full_path = PROJECT_PATH / rel_path
    if not full_path.exists():
        continue
    file_content = full_path.read_text(errors="ignore")
    for rule in rules:
        target = rule.get("target")
        if target:
            is_match = False
            for fnd in info["findings"]:
                if target in fnd:
                    is_match = True
                    break
            if not is_match:
                if rule.get("type") == "import_declaration":
                    if f"import {target}" in file_content:
                        is_match = True
                else: 
                    target_short = target.split()[-1].split("(")[0]
                    if target_short in file_content:
                        is_match = True
            if is_match and rule not in info["matching_rules"]:
                info["matching_rules"].append(rule)

print(f"\nFound {len(file_findings)} file(s) requiring translation:")
for path, info in file_findings.items():
    print(f"\n- File: {path}")
    print(f"  Findings:")
    for fnd in info["findings"]:
        print(f"    * {fnd}")
    print(f"  Matching Rules:")
    for r in info["matching_rules"]:
        print(f"    * [{r.get('rule_id')}] {r.get('target')} -> {r.get('new')} ({r.get('reason')})")

# Save diagnostic reports to target/ and artifacts/ so the agent can load them
for folder in ["artifacts", "target"]:
    dir_path = PROJECT_PATH / folder
    dir_path.mkdir(parents=True, exist_ok=True)
    with open(dir_path / "jdeprscan_report.json", "w", encoding="utf-8") as f:
        json.dump(updated_jdeprscan_result, f, indent=2, default=str)
    change_candidates = []
    for f_path, f_info in file_findings.items():
        change_candidates.append({
            "file_path": f_path,
            "reason": "; ".join(f_info["findings"]),
            "matching_rules": f_info["matching_rules"]
        })
    mygrate_report = {
        "status": "ok",
        "change_candidates": change_candidates
    }
    with open(dir_path / "mygrate_report.json", "w", encoding="utf-8") as f:
        json.dump(mygrate_report, f, indent=2, default=str)
print("\nSuccessfully saved jdeprscan_report.json and mygrate_report.json to target/ and artifacts/!")


Compiling project under JDK 17...
Initial compile status: 1
Running jdeprscan pipeline on the working codebase...
  [jdeprscan] Khám phá JDK...
  [jdeprscan] JDK 8:  C:\Users\tngtr\AppData\Local\Java\jdk-8
  [jdeprscan] JDK 17: C:\Program Files\Java\jdk-17
  [jdeprscan] jdeprscan: C:\Program Files\Java\jdk-17\bin\jdeprscan.exe
  [jdeprscan] Maven: C:\Users\tngtr\AppData\Local\Apache\apache-maven-3.9.16\bin\mvn
  [jdeprscan] B0: Maven resolve dependencies...
  [jdeprscan] B0: đang resolve dependencies...
  [jdeprscan] B0: Maven FAIL (exit code 1)
  [jdeprscan] B1: Compile project...
  [jdeprscan] B1: compiling 27 source files...
  [jdeprscan] B1: Compile FAIL — D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash\src\main\java\org\sonar\plugins\stash\IssuePathResolver.java:3: error: package org.sonar.api.batch.postjob.issue does not exist
impor
  [jdeprscan] B2: jdeprscan project JAR...
  [jdeprscan] B2: project JAR không có — skip
  [jdeprscan] B3: jdeprscan dependencies.

## 2. Iterative Translation Loop

We run a focused code translation loop to resolve JDK 17 migration blockers file-by-file.
For each file, we call ChatOllama directly to refactor code, apply the modified source, and check the Maven compilation status to verify correctness.

In [14]:
from src.agents.translator_agent_2 import TranslatorAgent_2
print("TranslatorAgent_2 class imported successfully from src.agents.translator_agent_2!")


TranslatorAgent_2 class imported successfully from src.agents.translator_agent_2!


In [15]:
import json
from pathlib import Path
from datetime import datetime

LOG_FILE = Path("log.txt")

print("=== STARTING DEDICATED CODE TRANSLATION LOOP ===")

from src.tools.maven import MavenRunner
runner = MavenRunner(target_java_version="17")
compile_res = runner.compile(PROJECT_PATH, clean=True)
compile_output = compile_res.stdout + "\n" + compile_res.stderr

# Clean and truncate initial compile output to avoid context window blowup and hangs
filtered_lines = []
for line in compile_output.splitlines():
    if len(line) > 800 or "-classpath" in line or " -d " in line or "-bootclasspath" in line:
        continue
    filtered_lines.append(line)
if len(filtered_lines) > 150:
    truncated_compile_output = (
        "\n".join(filtered_lines[:75]) +
        "\n... [TRUNCATED IN-BETWEEN LINES / CLASS PATH / DEBUG INFOS TO SAVE CONTEXT] ...\n" +
        "\n".join(filtered_lines[-75:])
    )
else:
    truncated_compile_output = "\n".join(filtered_lines)

print(f"\nCompile exit code: {compile_res.status}")
if compile_res.status == 0:
    print("✓ Project already compiles! No migration needed.")
else:
    payload = {
        "project_path": str(PROJECT_PATH),
        "target_java_version": "17",
        "current_instruction": (
            "Fix all the compilation errors in the project. "
            "Look at the compilation log in findings, open the files with errors using read_file, "
            "apply the edits using apply_edits or write_file (to overwrite if needed), "
            "and compile to verify until the project compiles cleanly."
        ),
        "findings": truncated_compile_output,
        "rules": ""
    }

    agent = TranslatorAgent_2()
    result = agent.run(json.dumps(payload, ensure_ascii=False, indent=2))

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n{'='*80}\n")
        f.write("PHASE 2 — ALL FIXES\n")
        f.write(f"TIME: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"{'='*80}\n\n")
        f.write("--- Compile Errors ---\n")
        f.write(compile_output + "\n\n")
        f.write("--- Tool Call Log ---\n")
        f.write(agent.get_log() + "\n\n")
        f.write("--- Final Result ---\n")
        f.write(str(result)[:3000] + "\n")
        f.write(f"\n---\n\n")

    print("→ Phase 2 complete. Check log.txt for detailed run history.")


=== STARTING DEDICATED CODE TRANSLATION LOOP ===

Compile exit code: 1
-> [LLM] ChatOllama initialised: model=gemma4:31b-cloud, base_url=https://ollama.com, auth=Bearer
-> [TRANSLATOR_2] ReAct iteration 1/30 [provider=ollama, model=gemma4:31b-cloud]
-> [TRANSLATOR_2] Calling tool: read_file({"file_path": "pom.xml"})
-> [TRANSLATOR_2] ReAct iteration 2/30 [provider=ollama, model=gemma4:31b-cloud]
-> [TRANSLATOR_2] Calling tool: check_maven_plugin({"artifact_id": "maven-jdeprscan-plugin", "group_id": "org.apache.maven.plugins", "version": "3.1.2"})
-> [TRANSLATOR_2] ReAct iteration 3/30 [provider=ollama, model=gemma4:31b-cloud]
-> [TRANSLATOR_2] Calling tool: apply_edits({"edits": [{"end_line": 279, "replacement": "<!--\n       <plugin>\n         <groupId>org.apache.maven.plugins</groupId>\n         <artifactId>maven-jdeprscan-plugin</artifactId>\n         <version>3.)
-> [TRANSLATOR_2] ReAct iteration 4/30 [provider=ollama, model=gemma4:31b-cloud]
-> [TRANSLATOR_2] Calling tool: compile

## 3. Final Verification & Secondary Checks

We verify that the whole codebase compiles successfully under JDK 17, and check remaining AST structures for libraries like Mockito.

In [16]:
# Run a final clean compilation with MavenRunner targeting Java 17 to verify all changes
runner = MavenRunner(target_java_version="17")
print("Running final clean mvn compile to verify project state...")
res = runner.compile(PROJECT_PATH, clean=True)
print(f"Compilation exit code: {res.status}")

if res.status == 0:
    print("Success! The project compiles cleanly under JDK 17 after translation and upgrades.")
else:
    print("Errors encountered during compile. Showing stdout/stderr output:")
    print("--- STDOUT ---")
    print(res.stdout)
    print("--- STDERR ---")
    print(res.stderr)

Running final clean mvn compile to verify project state...
Compilation exit code: 0
Success! The project compiles cleanly under JDK 17 after translation and upgrades.


In [17]:
# Find usages of Mockito in import statements using tree-sitter AST queries in the isolated directory
print("Querying AST for Mockito import declarations:")
mockito_imports = find_code_usages(
    project_path=str(PROJECT_PATH),
    node_type="import_declaration",
    identifier="mockito"
)

print(f"Found {len(mockito_imports)} Mockito import nodes:")
for imp in mockito_imports[:10]:
    print(f"  - {imp['file_path']} | Line {imp['start_line']}: {imp['snippet'].strip()}")

Querying AST for Mockito import declarations:
Found 44 Mockito import nodes:
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 5: import static org.mockito.ArgumentMatchers.any;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 6: import static org.mockito.ArgumentMatchers.anyString;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 7: import static org.mockito.ArgumentMatchers.eq;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 8: import static org.mockito.Mockito.mock;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 9: import static org.mockito.Mockito.times;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 10: import static org.mockito.Mockito.verify;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 11: import static org.mockito.Mockito

In [18]:
# Run the python agent unit tests to verify indexer and overlap validation
import subprocess
import sys
print("Running TranslatorAgent_2 unit tests...")
res = subprocess.run([sys.executable, "tests/test_translator_agent_2_tools.py"], capture_output=True, text=True)
print(f"Test exit code: {res.returncode}")
print("--- STDOUT ---")
print(res.stdout)
print("--- STDERR ---")
print(res.stderr)

Running TranslatorAgent_2 unit tests...
Test exit code: 1
--- STDOUT ---
-> [LLM] ChatOllama initialised: model=gemma4:31b-cloud, base_url=https://ollama.com, auth=Bearer
test_list_project_files PASSED

--- STDERR ---
Traceback (most recent call last):
  File "d:\capstone_project\MYGRATE---Capstone-Project\tests\test_translator_agent_2_tools.py", line 119, in <module>
    test_apply_edits_overlap(Path(td))
  File "d:\capstone_project\MYGRATE---Capstone-Project\tests\test_translator_agent_2_tools.py", line 47, in test_apply_edits_overlap
    project_dir.mkdir()
  File "C:\Users\tngtr\AppData\Local\Programs\Python\Python312\Lib\pathlib.py", line 1312, in mkdir
    os.mkdir(self, mode)
FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\tngtr\\AppData\\Local\\Temp\\tmp8s8lq50b\\project'



In [19]:
print("=== STEP 2 MIGRATION COMPLETION SUMMARY ===")
print(f"Code translation and JDK 17 migration completed in isolated folder!")
print(f"The migrated codebase is preserved at: {PROJECT_PATH}")

=== STEP 2 MIGRATION COMPLETION SUMMARY ===
Code translation and JDK 17 migration completed in isolated folder!
The migrated codebase is preserved at: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash
